# Raport: Etap 3 - Klasyfikacja Zero-Shot

## 1. Cel etapu
Celem tego etapu jest ocena zdolności pretrenowanego modelu językowego do klasyfikacji tweetów bez dodatkowego trenowania (zero-shot). Wykorzystujemy model `facebook/bart-large-mnli` — duży model transformerowy wytrenowany na zadaniu Natural Language Inference (NLI) — do przewidywania, czy dany tweet opisuje rzeczywistą katastrofę. Wynik stanowi punkt odniesienia (baseline) przed fine-tuningiem.

In [1]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score

c:\Projects\P2_Gr2_Basic\myvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Wczytanie danych
Wczytujemy wyczyszczony zbiór danych przygotowany w poprzednim etapie (EDA). Na potrzeby testów ograniczamy się do pierwszych 200 rekordów, aby przyspieszyć proces klasyfikacji przy użyciu modelu LLM.

In [6]:
df = pd.read_csv('../outputs/train_cleaned.csv').head(200)

## 3. Inicjalizacja modelu Zero-Shot
Używamy pipeline `zero-shot-classification` z biblioteki Hugging Face `transformers` z modelem `facebook/bart-large-mnli`. Jest to duży model BART wytrenowany na zbiorze MultiNLI (Natural Language Inference), który potrafi oceniać relacje semantyczne między tekstem a etykietami bez dodatkowego trenowania. Model ładowany jest na GPU (`device=0`), jeśli jest dostępne.

In [7]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 4779.52it/s]


## 4. Funkcja klasyfikacji
Definiujemy funkcję `predict_disaster`, która dla każdego tweeta wywołuje model z dwiema etykietami kandydującymi: `"real disaster"` oraz `"not a disaster"`. Model zwraca posortowaną listę etykiet według prawdopodobieństwa — jeśli na pierwszym miejscu znajdzie się `"real disaster"`, tweet zostaje oznaczony jako `1` (katastrofa), w przeciwnym razie jako `0`.

In [8]:
def predict_disaster(text):
    result = classifier(text, candidate_labels=["real disaster", "not a disaster"])
    return 1 if result['labels'][0] == "real disaster" else 0

## 5. Klasyfikacja zbioru danych i ocena wyników
Aplikujemy funkcję klasyfikacji na każdym wierszu kolumny `clean_text` i zapisujemy predykcje w nowej kolumnie `prediction`. Następnie porównujemy je z rzeczywistymi etykietami (`target`) przy użyciu metryki dokładności (Accuracy).

In [9]:
print("Rozpoczęcie klasyfikacji Zero-shot...")
df['prediction'] = df['clean_text'].apply(predict_disaster)

Rozpoczęcie klasyfikacji Zero-shot...


In [10]:
accuracy = accuracy_score(df['target'], df['prediction'])
print(f"Zero-shot Accuracy: {accuracy * 100:.2f}%")

Zero-shot Accuracy: 69.00%


**Analiza + interpretacja:**
Uzyskana dokładność (Accuracy) modelu zero-shot stanowi punkt odniesienia (baseline) dla kolejnych etapów projektu, w których model będzie dostrajany (fine-tuning) na danych treningowych. Wynik potwierdza, że model pretrenowany na zadaniu NLI potrafi w pewnym stopniu rozróżnić tweety katastroficzne od zwykłych, bazując wyłącznie na semantycznym znaczeniu tekstu — bez jakichkolwiek przykładów uczących. Ograniczenie do 200 próbek wynika z wysokiego kosztu obliczeniowego modelu; pełna ewaluacja będzie przeprowadzona po fine-tuningu.
